# Adım 3: Spark Structured Streaming + Delta Lake

Bu notebook'ta:
- Kafka'dan gelen loan-events akışı okunur
- **Bronze** katmanına ham veri yazılır
- **Silver** katmanında veri temizlenir
- **Gold** katmanında analiz için hazır tablo oluşturulur

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, from_json, to_timestamp, when, regexp_replace, trim, count, isnan
from pyspark.sql.types import StructType, StructField, StringType, DoubleType

spark = (
    SparkSession.builder
    .appName("CreditRisk-Step3")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    .config("spark.jars.packages",
            "io.delta:delta-spark_2.12:3.0.0,"
            "org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.0")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")
print("Spark Version:", spark.version)

## Şema Tanımı

In [ ]:
LOAN_SCHEMA = StructType([
    StructField("timestamp",           StringType()),
    StructField("loan_id",             StringType()),
    StructField("event_type",          StringType()),
    StructField("loan_amnt",           DoubleType()),
    StructField("funded_amnt",         DoubleType()),
    StructField("term",                StringType()),
    StructField("int_rate",            StringType()),
    StructField("installment",         DoubleType()),
    StructField("grade",               StringType()),
    StructField("sub_grade",           StringType()),
    StructField("emp_length",          StringType()),
    StructField("home_ownership",      StringType()),
    StructField("annual_inc",          DoubleType()),
    StructField("verification_status", StringType()),
    StructField("issue_d",             StringType()),
    StructField("loan_status",         StringType()),
    StructField("purpose",             StringType()),
    StructField("dti",                 DoubleType()),
    StructField("delinq_2yrs",         DoubleType()),
    StructField("open_acc",            DoubleType()),
    StructField("pub_rec",             DoubleType()),
    StructField("revol_bal",           DoubleType()),
    StructField("revol_util",          StringType()),
    StructField("total_acc",           DoubleType()),
])

## Kafka'dan Okuma & Bronze Katmanı

In [ ]:
KAFKA_BOOTSTRAP = "kafka:29092"
KAFKA_TOPIC = "loan-events"
DELTA_BASE = "/app/delta_lake"

raw_df = (
    spark.readStream
    .format("kafka")
    .option("kafka.bootstrap.servers", KAFKA_BOOTSTRAP)
    .option("subscribe", KAFKA_TOPIC)
    .option("startingOffsets", "earliest")
    .load()
    .selectExpr("CAST(value AS STRING) as raw_json")
    .select(from_json(col("raw_json"), LOAN_SCHEMA).alias("data"))
    .select("data.*")
)

bronze_query = (
    raw_df.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", f"{DELTA_BASE}/checkpoints/bronze")
    .trigger(processingTime="10 seconds")
    .start(f"{DELTA_BASE}/bronze/loans")
)

print("Bronze streaming başladı.")
bronze_query.awaitTermination(60)

## Silver Katmanı — Veri Temizleme

In [ ]:
bronze_df = spark.read.format("delta").load(f"{DELTA_BASE}/bronze/loans")
print(f"Bronze kayıt sayısı: {bronze_df.count():,}")
bronze_df.printSchema()

In [ ]:
# Null analizi
null_counts = bronze_df.select([
    count(when(col(c).isNull(), c)).alias(c) for c in bronze_df.columns
])
null_counts.show()

In [ ]:
silver_df = (
    bronze_df
    .filter(col("loan_id").isNotNull())
    .filter(col("loan_amnt") > 0)
    .filter(col("annual_inc") > 0)
    .filter(col("loan_status").isNotNull())
    .dropDuplicates(["loan_id"])
    .withColumn("int_rate_pct",
                regexp_replace(trim(col("int_rate")), "%", "").cast("double"))
    .withColumn("revol_util_pct",
                regexp_replace(trim(col("revol_util")), "%", "").cast("double"))
    .withColumn("event_time", to_timestamp(col("timestamp")))
    .withColumn("is_default", when(
        col("loan_status").isin("Charged Off", "Default", "Late (31-120 days)"), 1
    ).otherwise(0))
    .drop("int_rate", "revol_util", "timestamp")
)

print(f"Silver kayıt sayısı: {silver_df.count():,}")
print(f"Temerrüt oranı: {silver_df.filter(col('is_default')==1).count() / silver_df.count():.2%}")

silver_df.write.format("delta").mode("overwrite").save(f"{DELTA_BASE}/silver/loans")
print("Silver katmanına yazıldı.")

## Gold Katmanı — Analiz için Hazır Tablo

In [ ]:
from pyspark.sql.functions import year, month

gold_df = (
    silver_df
    .filter(col("loan_status").isin(
        "Fully Paid", "Charged Off", "Default", "Late (31-120 days)"
    ))
    .withColumn("issue_year",  year(to_timestamp(col("issue_d"), "MMM-yyyy")))
    .withColumn("issue_month", month(to_timestamp(col("issue_d"), "MMM-yyyy")))
    .withColumn("term_months",
                regexp_replace(trim(col("term")), " months", "").cast("integer"))
)

print(f"Gold kayıt sayısı: {gold_df.count():,}")
gold_df.write.format("delta").mode("overwrite").save(f"{DELTA_BASE}/gold/loans")
print("Gold katmanına yazıldı.")

## Delta Lake Doğrulama

In [ ]:
gold = spark.read.format("delta").load(f"{DELTA_BASE}/gold/loans")
print("Gold tablo şeması:")
gold.printSchema()
gold.show(5)

In [ ]:
gold.groupBy("loan_status").count().orderBy("count", ascending=False).show()